### First: Categorize Universe level Data Entries 

In [0]:
%sql
WITH raw_data AS (
  SELECT DISTINCT
    ev3.universe,
    UPPER(TRIM(ev3.table)) AS BO_TABLE,
    CASE
      WHEN UPPER(ev3.SOURCE) LIKE 'ALIAS%' AND ev3.base_table IS NOT NULL AND ev3.base_table != ''
        THEN UPPER(TRIM(ev3.base_table))
      ELSE UPPER(TRIM(ev3.table))
    END AS source_table,
    UPPER(TRIM(ev3.schema)) AS SCHEMA,
    CASE
      WHEN UPPER(ev3.SOURCE) LIKE 'ALIAS%'
        THEN UPPER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(ev3.SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE UPPER(TRIM(ev3.SOURCE))
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction as ev3

  UNION ALL

  SELECT  DISTINCT
    ev4.universe,
    UPPER(TRIM(ev4.table)) AS BO_TABLE,
    CASE
      WHEN UPPER(ev4.SOURCE) LIKE 'ALIAS%' AND ev4.base_table IS NOT NULL AND ev4.base_table != ''
        THEN UPPER(TRIM(ev4.base_table))
      ELSE UPPER(TRIM(ev4.table))
    END AS source_table,
    UPPER(TRIM(ev4.schema)) AS SCHEMA,
    CASE
      WHEN UPPER(ev4.SOURCE) LIKE 'ALIAS%'
        THEN UPPER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(ev4.SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE UPPER(TRIM(ev4.SOURCE))
    END AS SOURCE

  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining as ev4
), 
ranked_source AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY universe, BO_TABLE, SOURCE_TABLE, SCHEMA
      ORDER BY
        CASE 
          WHEN SOURCE = 'SYMPHONY' THEN 1
          WHEN SOURCE = 'ORACLE FINANCE' THEN 2
          WHEN SOURCE = 'ORACLE FINANCE TABLE' THEN 3
          WHEN SOURCE = 'ORACLE FINANCE VIEW' THEN 4
          WHEN SOURCE = 'ORACLEFINANCE' THEN 5
          WHEN SOURCE = 'ORACLE FINANCE MATERIALIZED VIEW' THEN 6
          WHEN SOURCE = 'ORACLE FINACNE' THEN 7
          WHEN SOURCE = 'DW (FACT)' THEN 8
          WHEN SOURCE = 'DW' THEN 9
          WHEN SOURCE = 'DW (DIMENSION)' THEN 10
          WHEN SOURCE = 'DW (VIEW)' THEN 11
          WHEN SOURCE = 'INFORMATICA' THEN 12
          WHEN SOURCE = 'SYSTEM TABLE' THEN 13
          WHEN SOURCE = 'DERIVED' THEN 14
          WHEN SOURCE IN ('NA', 'NOT AVAILABLE') THEN 15
          ELSE 99
        END
    ) AS rn1
  FROM raw_data
),
deduped_source AS (
  SELECT universe, BO_TABLE, SOURCE_TABLE, SCHEMA, SOURCE
  FROM ranked_source
  WHERE rn1 = 1
),
-- Level 2: Deduplicate by (BO_TABLE, SOURCE_TABLE) — keep highest priority SCHEMA
ranked_schema AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY BO_TABLE, SOURCE_TABLE
      ORDER BY
        CASE 
          WHEN SCHEMA = 'ORABUP0' THEN 1
          WHEN SCHEMA = 'ORADMART1' THEN 2
          WHEN SCHEMA = 'AR' THEN 3
          WHEN SCHEMA = 'GL' THEN 4
          WHEN SCHEMA = 'APPS' THEN 5
          WHEN SCHEMA = 'AP' THEN 6
          WHEN SCHEMA = 'CUST' THEN 7
          WHEN SCHEMA = 'FA' THEN 8
          WHEN SCHEMA = 'APPLSYS' THEN 9
          WHEN SCHEMA = 'PO' THEN 10
          WHEN SCHEMA = 'HR' THEN 11
          WHEN SCHEMA = 'XLA' THEN 12
          WHEN SCHEMA = 'JTF' THEN 13
          WHEN SCHEMA = 'INV' THEN 14
          WHEN SCHEMA = 'ZX' THEN 15
          WHEN SCHEMA = 'OKC' THEN 16
          WHEN SCHEMA = 'ICX' THEN 17
          WHEN SCHEMA = 'SYS' THEN 18
          WHEN SCHEMA = 'ORABOFP' THEN 19
          WHEN SCHEMA = 'GBRELS1' THEN 20
          WHEN SCHEMA IN ('NOT AVAILABLE', 'NOT AVILABLE') THEN 21
          ELSE 99
        END
    ) AS rn2
  FROM deduped_source
),
Universe_list as (
  SELECT universe, BO_TABLE, SOURCE_TABLE, SCHEMA, SOURCE
  FROM ranked_schema
  WHERE rn2 = 1
  ORDER BY universe, BO_TABLE, SOURCE_TABLE ASC
)
select 
  upper(trim(Universe_list.universe)) as UNIVERSE,
  upper(trim(Universe_list.BO_TABLE)) as BO_SQL_TABLE, 
  upper(trim(Universe_list.SOURCE_TABLE)) as MI_Table, 
  upper(trim(Universe_list.SCHEMA)) as MI_SCHEMA, 
  upper(trim(Universe_list.SOURCE)) as MI_SOURCE, 
  CASE
    WHEN upper(trim(Universe_list.SOURCE)) = 'SYMPHONY' 
      THEN upper(trim(Universe_list.SOURCE_TABLE))
    WHEN upper(trim(Universe_list.SOURCE)) LIKE '%ORACLE%' 
      THEN upper(trim(Universe_list.SOURCE_TABLE))
    ELSE COALESCE(upper(trim(d1.SOURCE_TABLE)), upper(trim(Universe_list.SOURCE_TABLE)))
  END as final_table,
  CASE
    WHEN upper(trim(Universe_list.SOURCE)) = 'SYMPHONY' 
      THEN 'SYMPHONY'
    WHEN upper(trim(Universe_list.SOURCE)) LIKE '%ORACLE%' 
      THEN upper(trim(Universe_list.SCHEMA))
    ELSE COALESCE(upper(trim(d1.SOURCE_SCHEMA)), upper(trim(Universe_list.SCHEMA)))
  END as final_schema
from Universe_list 
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.MI_Dictionary_SAP_BO as D1
on upper(trim(Universe_list.SOURCE_TABLE)) = upper(trim(D1.TARGET_TABLE)) 
   and upper(trim(Universe_list.SCHEMA)) = upper(trim(D1.TARGET_SCHEMA))
order by upper(trim(Universe_list.BO_TABLE)) asc, upper(trim(Universe_list.SOURCE_TABLE)) asc


In [0]:
from pyspark.sql import functions as F

# Use the SQL result from cell 2 (_sqldf)
mi_tables = _sqldf.select(
    "UNIVERSE", "BO_SQL_TABLE", "MI_Table", "MI_SCHEMA", "MI_SOURCE",
    "final_table", "final_schema"
).distinct()

# ─────────────────────────────────────────────────────────────────────────────
# CATEGORIZATION — 4 rules applied in priority order:
#
#  Rule 1 │ MI_SOURCE contains 'ORACLE'  → Finance / Accounting  (direct rule)
#  Rule 2 │ MI_SOURCE = 'SYMPHONY'       → final_table prefix pattern
#          │   TB** prefix = business domain coded in Symphony table naming
#          │   VW** prefix = views follow same prefix convention
#  Rule 3 │ UNIVERSE name keywords       → business context for DW/DERIVED rows
#          │   Universe name describes the report's business purpose,
#          │   so same-family universes (e.g. POLICY BSB, POLICY MONTHLY)
#          │   map to the same category
#  Rule 4 │ final_table name fallback    → last-resort pattern match
# ─────────────────────────────────────────────────────────────────────────────

categorized_final = mi_tables.withColumn("SUGGESTED_CATEGORY",

    # ── RULE 1: Oracle Finance source OR Oracle Finance schema → Finance / Accounting ──
    # MI_SOURCE: ORACLE FINANCE, ORACLE FINANCE TABLE/VIEW, ORACLEFINANCE, typos
    # final_schema: ORFM (Oracle Financial Mgmt), ORFP (Oracle Financial Planning)
    F.when(
        F.col("MI_SOURCE").rlike("(?i)ORACLE") |
        F.col("final_schema").isin("ORFM", "ORFP"),
        "Finance / Accounting"
    )

    # ── RULE 2: Symphony table prefix patterns ───────────────────────────────
    # Claims: TBCL_ = Claims tables; TBBO_CLAIM* = bond claim sub-tables
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBCL_|TBBO_CLAIM)"), "Claims / Collections")
    # Bonds/Surety: remaining TBBO_ (after claims sub-tables excluded above)
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^TBBO_"), "Bonds / Surety")
    # Policy / Underwriting: TBPO_ = Policy, TBAU_ = All-Credit/UW, VWPO_ = Policy views
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBPO_|TBAU_|VWPO_)"), "Policy / Underwriting")
    # Finance / Accounting: TBPA_ = Payments/Accounting, TBIS_ = Invoices/Statements
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBPA_|VWPA_|TBIS_|ALL_INVOICES)"), "Finance / Accounting")
    # Premium Management: TBPM_ = Premium calculation tables
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").startswith("TBPM_"), "Premium Management")
    # Buyer / Debtor: TBBU_ = Buyer management, VWBU_ = Buyer views
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBBU_|VWBU_)"), "Buyer / Debtor")
    # Sales / Commercial: TBSA_ = Sales, TBMK_ = Marketing, VWSA_ = Sales views
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBSA_|TBMK_|VWSA_)"), "Sales / Commercial")
    # Workflow / User Management: TBWM_ = Workflow, TBSE_ = Security, VWND_ = Auth alerts
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBWM_|TBSE_|VWND_)"), "Workflow / User Management")
    # Customer Service: TBCS_ = Customer Service tables
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").startswith("TBCS_"), "Customer Service")
    # Communications: TBCM_ = Correspondence, TBDO_ = Letter/Text definitions
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBCM_|TBDO_)"), "Communications / Correspondence")
    # Medium-Term / Special Products: TBMT_ = Medium-Term, VWMT_ views
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBMT_|VWMT_)"), "Medium-Term / Special Products")
    # Asset Management: TBAM_ = Asset monitoring tables
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").startswith("TBAM_"), "Asset Management / Monitoring")
    # Risk / Credit Assessment: TBMI_ = Market Intelligence/Rating, VWMI_ views
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBMI_|VWMI_)"), "Risk / Credit Assessment")
    # Reinsurance / Cabrillo: TBGG_ = Group/Guarantee reinsurance tables
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").startswith("TBGG_"), "Reinsurance / Cabrillo")
    # ERP / System Config: TBEA_ = Enterprise App, VWND_ already handled above
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").startswith("TBEA_"), "ERP / System Config")
    # Organisation / Reference Data: TBOR_ = Org master data, CG_REF* = code groups,
    #   TBEN_ = Business Entities (org roles, contacts), VW_* and VWOR_ = org views
    .when((F.col("MI_SOURCE") == "SYMPHONY") &
          F.col("final_table").rlike("^(TBOR_|CG_RE|TBEN_|VWOR_|VW_)"), "Organisation / Reference Data")

    # ── RULE 3: Universe name keywords (DW / DERIVED / INFORMATICA / NA) ──────
    # Finance / Accounting — universe names referencing financial processes
    .when(F.col("UNIVERSE").rlike(
        "(?i)(FINANC|AP.AR.GL|PAYABL|RECEIVABL|AGING|PRUDENCE|PAYMENT.*RECOV|PAYMT.*RECOV"
        "|INVOICE|LEDGER|BUDGET|SCHED.2|SPANISH.TAX|EKV|CUMULATIVE.CASES.FIN"
        "|P2P|CONTROL.REPORT|EXCHANGE.RATE|UPR.DETAILED)"
    ), "Finance / Accounting")
    # Claims — universe names referencing claims, overdues, collections
    .when(F.col("UNIVERSE").rlike(
        "(?i)(CLAIM|OVERDUE|COLLECTION|GESTION.RECOUVR|INITIAL.CLAIMS|GKS.CLAIMS)"
    ), "Claims / Collections")
    # Policy / Underwriting
    .when(F.col("UNIVERSE").rlike(
        "(?i)(POLICY|POLIC|UNDERWRITING|MODULE.POLICIES|NPP.CORE|COVER|MEXICO|RECONCILIATION_GKS)"
    ), "Policy / Underwriting")
    # Buyer / Debtor — credit limits processed/approved, buyer management
    .when(F.col("UNIVERSE").rlike(
        "(?i)(CREDIT.LIMIT|BUYER|DEBTOR|SMALL.CREDIT|ALL.CREDIT|CREDIT.LIMITS|CREDIT.LIMIT.EXPOSURE)"
    ), "Buyer / Debtor")
    # Sales / Commercial — sales pipelines, lead management, Arcade
    .when(F.col("UNIVERSE").rlike(
        "(?i)(ARCADE.OFFER|SALES|COMMERCIAL|LEAD.MANAGEMENT|PIPELINE_PREMIUM)"
    ), "Sales / Commercial")
    # Premium Management — pricing, premium pipeline
    .when(F.col("UNIVERSE").rlike(
        "(?i)(PRICING|PIPELINE_PREMIUM)"
    ), "Premium Management")
    # Bonds / Surety
    .when(F.col("UNIVERSE").rlike(
        "(?i)(BOND|SURETY|GUARANTEE)"
    ), "Bonds / Surety")
    # Customer Service
    .when(F.col("UNIVERSE").rlike(
        "(?i)(CUSTOMER.SERVICE|PASSPORT)"
    ), "Customer Service")
    # Reinsurance / Cabrillo
    .when(F.col("UNIVERSE").rlike(
        "(?i)(CABRILLO|COMBINED.*CLAIMS.*FINANCE|CLAIMS.*FINANCE|UPR.DETAILED)"
    ), "Reinsurance / Cabrillo")
    # Medium-Term / Special Products
    .when(F.col("UNIVERSE").rlike(
        "(?i)(MEDIUM.TERM|SINGLE.RISK)"
    ), "Medium-Term / Special Products")
    # Workflow / User Management
    .when(F.col("UNIVERSE").rlike(
        "(?i)(USER.SECURITY|WORKFLOW.MI|WFR.TOOLS)"
    ), "Workflow / User Management")
    # Asset Management
    .when(F.col("UNIVERSE").rlike(
        "(?i)(AM.TOOL|ASSET.MGMT)"
    ), "Asset Management / Monitoring")
    # Organisation / Reference Data — TPE = Trading Partner Entities, REF_ISO
    .when(F.col("UNIVERSE").rlike(
        "(?i)(REFERENCE.DATA|ORG.COMM|TPE|ABRP|REF_ISO)"
    ), "Organisation / Reference Data")
    # WITS / IT Service Management
    .when(F.col("UNIVERSE").rlike(
        "(?i)(^WITS$|QUALITY.CENTER|KPI)"
    ), "WITS / IT Service Management")

    # ── RULE 4: final_table name fallback ─────────────────────────────────────
    .when(F.col("final_table").rlike(
        "(?i)(^GL_|^AP_|^AR_|^XLA_|INVOICE|PAYMENT|RECEIPT|LEDGER|BALANCE"
        "|JOURNAL|AGING|AGED_|DSO_|^RA_CUST|RECEIVABLE|WEIGHTED|PRUDENCE)"
    ), "Finance / Accounting")
    .when(F.col("final_table").rlike(
        "(?i)(CLAIM|RECOVERY|OVERDUE|DEBT|COLLECTION|RECOURSE)"
    ), "Claims / Collections")
    .when(F.col("final_table").rlike(
        "(?i)(POLICY|PREMIUM|COVER|EXPOSURE|ENDORSEMENT)"
    ), "Policy / Underwriting")
    .when(F.col("final_table").rlike(
        "(?i)(BUYER|CREDIT.LIMIT|SUBLIMIT|GRADE)"
    ), "Buyer / Debtor")
    .when(F.col("final_table").rlike(
        "(?i)(^DD_|^DIM_|^FACT_|^FT_|^FC_|^DW_|^AGG_|^MART_|AGED_STEERING|AGING_STEERING)"
    ), "Data Warehouse / Dimensions")
    .when(F.col("final_table").rlike(
        "(?i)(^FND_|LDAP|LOOKUP|PARAMETER|CONFIG|SYSTEM_TABLE)"
    ), "ERP / System Config")
    .when(F.col("final_table").rlike(
        "(?i)(USER|SECURITY|WORKGRP|DEPT|SECTION|AGENT)"
    ), "Workflow / User Management")
    .otherwise("Other / Unclassified")
)

# Write to Unity Catalog
target_table = "dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_table_category"

categorized_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"✓ Written {categorized_final.count()} rows to {target_table}")
print()
print("=== CATEGORY DISTRIBUTION ===")
display(
    spark.table(target_table)
    .groupBy("SUGGESTED_CATEGORY")
    .agg(
        F.countDistinct("final_table").alias("distinct_final_tables"),
        F.countDistinct("UNIVERSE").alias("distinct_universes"),
        F.count("*").alias("total_rows")
    )
    .orderBy(F.col("total_rows").desc())
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load data items and category lookup (prepared by cell 3)
data_items_raw = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition")
category = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_table_category")

# Clean universe_name: strip '_old_udt', '.unx', and 'Obsolete' suffixes
data_items = data_items_raw \
    .withColumn("universe_name_clean",
        F.trim(F.regexp_replace(F.col("universe_name"), r"(_old_udt|\.unx|\s*Obsolete)$", ""))
    )

# --- Extract BO table alias and schema from SQL_Tables_str ---
data_items_enriched = data_items \
    .withColumn("sql_table_extracted",
        F.upper(F.trim(
            F.when(F.col("SQL_Tables_str").contains("."),
                # Has schema prefix: extract table name after first dot,
                # stop at space/pipe/comma so multi-table strings yield only the first table
                F.regexp_extract(F.col("SQL_Tables_str"), r"[^.]+\.([^.,| ]+)", 1)
            ).otherwise(
                # No schema prefix: BO alias name — extract first token only
                F.regexp_extract(F.col("SQL_Tables_str"), r"^([^|, ]+)", 1)
            )
        ))
    ) \
    .withColumn("sql_schema_extracted",
        F.when(F.col("SQL_Tables_str").contains("."),
            F.upper(F.trim(F.regexp_extract(F.col("SQL_Tables_str"), r"^([^.]+)\.", 1)))
        ).otherwise(F.lit(None))
    )

# ── PRIMARY LOOKUP: (UNIVERSE, BO_SQL_TABLE) → SUGGESTED_CATEGORY ────────────
# Cell 3 prepares universe_table_category at (UNIVERSE, BO_SQL_TABLE) granularity.
# sql_table_extracted from SQL_Tables_str matches BO_SQL_TABLE (the BO alias name).
# This is the precise item-level join — same final_table can have different categories
# depending on the universe context it is used in.
precise_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(
            F.upper(F.trim(F.col("UNIVERSE"))),
            F.upper(F.trim(F.col("BO_SQL_TABLE")))
        ).orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("UNIVERSE"))).alias("cat_universe"),
        F.upper(F.trim(F.col("BO_SQL_TABLE"))).alias("cat_bo_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_primary")
    )

# ── FALLBACK LOOKUP: (UNIVERSE, final_table) → SUGGESTED_CATEGORY ────────────
# Catches schema.table format in SQL_Tables_str (e.g. "APPS.GL_CODE_COMBINATIONS")
# where extracted name equals the actual Oracle table name, not the BO alias.
# e.g. sql_table_extracted="GL_CODE_COMBINATIONS" → final_table="GL_CODE_COMBINATIONS" ✓
final_table_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(
            F.upper(F.trim(F.col("UNIVERSE"))),
            F.upper(F.trim(F.col("final_table")))
        ).orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("UNIVERSE"))).alias("fb_universe"),
        F.upper(F.trim(F.col("final_table"))).alias("fb_final_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_fallback")
    )

# ── FALLBACK 3: Cross-universe BO_SQL_TABLE lookup (no universe constraint) ─────
# For tables that exist in the category but under a DIFFERENT universe than where
# the data item lives. Since category is driven by table source (MI_SOURCE/prefix),
# the same table always gets the same category regardless of which universe uses it.
cross_univ_bo_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(F.upper(F.trim(F.col("BO_SQL_TABLE"))))
        .orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("BO_SQL_TABLE"))).alias("cx_bo_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_cross_univ")
    )

# ── FALLBACK 4: Universe name keyword matching ───────────────────────────────
# Last resort for: (A) NULL SQL_Tables_str (calculated items with no table ref)
# and (C) tables not in the category table at all.
univ_keyword_expr = (
    F.when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(FINANC|AP.AR.GL|PAYABL|RECEIVABL|AGING|PRUDENCE|PAYMT|PAYMENT.*RECOV"
        "|INVOICE|LEDGER|BUDGET|P2P|EXCHANGE.RATE|SCHED.2|SPANISH.TAX|EKV|CUMULATIVE.CASES.FIN)"
    ), F.lit("Finance / Accounting"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(CLAIM|OVERDUE|COLLECTION|GESTION.RECOUVR|INITIAL.CLAIMS|GKS.CLAIMS)"
    ), F.lit("Claims / Collections"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(POLICY|POLIC|UNDERWRITING|MODULE.POLICIES|NPP.CORE|COVER|MEXICO|RECONCILIATION_GKS)"
    ), F.lit("Policy / Underwriting"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(CREDIT.LIMIT|BUYER|DEBTOR|SMALL.CREDIT|ALL.CREDIT)"
    ), F.lit("Buyer / Debtor"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(ARCADE.OFFER|SALES|COMMERCIAL|LEAD.MANAGEMENT)"
    ), F.lit("Sales / Commercial"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(PRICING|PIPELINE_PREMIUM)"
    ), F.lit("Premium Management"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(BOND|SURETY|GUARANTEE)"
    ), F.lit("Bonds / Surety"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(CUSTOMER.SERVICE|PASSPORT)"
    ), F.lit("Customer Service"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(CABRILLO|COMBINED.*CLAIMS.*FINANCE|UPR.DETAILED|UPR.DAC)"
    ), F.lit("Reinsurance / Cabrillo"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(MEDIUM.TERM|SINGLE.RISK)"
    ), F.lit("Medium-Term / Special Products"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(USER.SECURITY|WORKFLOW|WFR.TOOLS|GROUP.HR)"
    ), F.lit("Workflow / User Management"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(AM.TOOL|ASSET.MGMT)"
    ), F.lit("Asset Management / Monitoring"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(REFERENCE.DATA|ORG.COMM|ORGANISATION|TPE|ABRP|REF_ISO|BELGIUM.CUST)"
    ), F.lit("Organisation / Reference Data"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(^WITS|QUALITY.CENTER|KPI)"
    ), F.lit("WITS / IT Service Management"))
    .when(F.upper(F.col("universe_name_clean")).rlike(
        "(?i)(FHSQL|SQL_INVENTORY|DALLAS|BBT_DWH|PVDATAMART|PVPORT|OID.PRODUCTION)"
    ), F.lit("Data Warehouse / Dimensions"))
)

# --- Manual mapping for universe names not covered by the category table ---
manual_universe_mapping = {
    # Data Warehouse / Dimensions
    "DALLAS": "Data Warehouse / Dimensions",
    "DALLAS2": "Data Warehouse / Dimensions",
    "BBT_DWH_BO_UNIVERSE": "Data Warehouse / Dimensions",
    "PVDATAMART ATRADIUS": "Data Warehouse / Dimensions",
    "PVDATAMART ATRADIUS NEW": "Data Warehouse / Dimensions",
    "PVPORT": "Data Warehouse / Dimensions",
    "OID PRODUCTION REPORTING": "Data Warehouse / Dimensions",
    "PRODUCTION REPOS UNIVERSE": "Data Warehouse / Dimensions",
    # Finance / Accounting
    "COMBINED AP AR GL WITH XLA": "Finance / Accounting",
    "COMBINED AP,AR & GL UNIVERSE": "Finance / Accounting",
    "FINANCIAL TRANSACTIONS": "Finance / Accounting",
    "PAYABLES": "Finance / Accounting",
    "COMBINED PAYM'T & RECOVERY": "Finance / Accounting",
    "COMBINED PAYM'T _ RECOVERY": "Finance / Accounting",
    "PAYM'T _ RECOVERY FIN": "Finance / Accounting",
    "PAYMT_RECOVERYFIN_DAILY": "Finance / Accounting",
    "EKV 3.2P": "Finance / Accounting",
    "AMT5": "Finance / Accounting",
    "PVDM_BUDGET": "Finance / Accounting",
    "SCHED-2": "Finance / Accounting",
    "SPANISH TAX": "Finance / Accounting",
    "CUMULATIVE CASES FIN_POL YR": "Finance / Accounting",
    # Policy / Underwriting
    "UNDERWRITING MONTHLY BSB": "Policy / Underwriting",
    "UNDERWRITING MONTHLY BSB ADE1": "Policy / Underwriting",
    "POLICY MONTHLY BSB": "Policy / Underwriting",
    "NPP CORE BSB": "Policy / Underwriting",
    "NPP CORE MONTHLY BSB": "Policy / Underwriting",
    "MODULE POLICIES MONTHLY BSB": "Policy / Underwriting",
    "NEW EXPOSURE SKELETON": "Policy / Underwriting",
    # Buyer / Debtor
    "CREDIT LIMITS PROCESSED MONTHLY": "Buyer / Debtor",
    # Claims / Collections
    "CER CLAIMS": "Claims / Collections",
    "GESTION RECOUVREMENT": "Claims / Collections",
    "OVERDUES CLAIMS MONTHLY BSB": "Claims / Collections",
    # WITS / IT Service Management
    "WITS CONTRACTS AND PURCHASING": "WITS / IT Service Management",
    "QUALITY CENTER": "WITS / IT Service Management",
    # ERP / System Config
    "FHSQL034": "Other / Unclassified",
    "FHSQL036": "Other / Unclassified",
    "FHSQL041": "Other / Unclassified",
    "FHSQL043": "Other / Unclassified",
    "FHSQL046": "Other / Unclassified",
    "FHSQL047": "Other / Unclassified",
    "FHSQL048": "Other / Unclassified",
    "FHSQL050": "Other / Unclassified",
    "FHSQL050 (2)": "Other / Unclassified",
    "FHSQL051": "Other / Unclassified",
    "FHSQL052 (2)": "Other / Unclassified",
    "SQL_INVENTORY": "Other / Unclassified",
    # Workflow / User Management
    "GROUP HR": "Workflow / User Management",
    # Organisation / Reference Data
    "TPE": "Organisation / Reference Data",
    "BELGIUM CUSTOMERS": "Organisation / Reference Data",
    # Customer Service
    "CUSTOMER SERVICE CENTER BY TYPE": "Customer Service",
    # Reinsurance / Cabrillo
    "UPR DAC": "Reinsurance / Cabrillo",
}

manual_mapping_expr = F.lit(None).cast("string")
for name, cat in manual_universe_mapping.items():
    manual_mapping_expr = F.when(
        F.upper(F.trim(F.col("universe_name_clean"))) == name, F.lit(cat)
    ).otherwise(manual_mapping_expr)

# --- Multi-level join and coalesce ---
    # Primary: (universe_name_clean, sql_table_extracted) → (UNIVERSE, BO_SQL_TABLE)
    # sql_table_extracted holds the BO alias name (no-dot SQL_Tables_str values like
    # "A1_TBMI_CSI_ELEMENT_TYPES") which matches BO_SQL_TABLE in universe_table_category
    # Fallback: (universe_name_clean, sql_table_extracted) → (UNIVERSE, final_table)
    # Catches schema.table SQL_Tables_str (e.g. "APPS.GL_CODE_COMBINATIONS") where the
    # extracted part after the dot equals final_table, not the BO alias

data_items_categorized = data_items_enriched \
    .join(precise_lookup,
        (F.upper(F.trim(F.col("universe_name_clean"))) == F.col("cat_universe")) &
        (F.col("sql_table_extracted") == F.col("cat_bo_table")),
        "left"
    ) \
    .join(final_table_lookup,
        (F.upper(F.trim(F.col("universe_name_clean"))) == F.col("fb_universe")) &
        (F.col("sql_table_extracted") == F.col("fb_final_table")),
        "left"
    ) \
    .join(cross_univ_bo_lookup,
        F.col("sql_table_extracted") == F.col("cx_bo_table"),
        "left"
    ) \
    .withColumn("manual_cat", manual_mapping_expr) \
    .withColumn("univ_keyword_cat", univ_keyword_expr) \
    .withColumn("SUGGESTED_CATEGORY",
        F.coalesce(
            # Primary: precise (universe + BO alias) → category from cell 3 lookup
            F.col("cat_primary"),
            # Fallback 1: (universe + final_table) for Oracle schema.table references
            F.col("cat_fallback"),
            # Fallback 2: ORFM/ORFP schema in SQL_Tables_str → Finance / Accounting
            F.when(F.col("sql_schema_extracted").isin("ORFM", "ORFP"),
                   F.lit("Finance / Accounting")),
            # Fallback 3: cross-universe BO_SQL_TABLE — same table, different universe context
            F.col("cat_cross_univ"),
            # Fallback 4: explicit manual mapping for known universe names
            F.col("manual_cat"),
            # Fallback 5: universe name keywords — last resort for NULL SQL_Tables_str
            #             or tables not captured in the category table at all
            F.col("univ_keyword_cat")
        )
    ) \
    .select(
        data_items_enriched["DataItem_unique_Id"],
        data_items_enriched["universe_id"],
        data_items_enriched["DataItem_id"],
        data_items_enriched["DataItem_name"],
        data_items_enriched["universe_name"],
        data_items_enriched["DataItem_type"],
        data_items_enriched["DataItem_description"],
        data_items_enriched["DataItem_dataType"],
        data_items_enriched["DataItem_path"],
        data_items_enriched["universe_path"],
        data_items_enriched["universe_type"],
        data_items_enriched["sql_definition"],
        data_items_enriched["DataItem_name_definition"],
        data_items_enriched["DataItem_path_definition"],
        data_items_enriched["SQL_Tables_str"],
        F.col("SUGGESTED_CATEGORY")
    )

# Write result
target = "dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition"
data_items_categorized.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)

# Summary
total = data_items_categorized.count()
matched = data_items_categorized.filter(F.col("SUGGESTED_CATEGORY").isNotNull()).count()
print(f"✓ Written {total:,} rows to {target}")
print(f"  Category filled: {matched:,} ({100*matched/total:.1f}%)")
print(f"  Category NULL:   {total-matched:,} ({100*(total-matched)/total:.1f}%)")
print(f"\n=== CATEGORY DISTRIBUTION ===")
display(
    data_items_categorized.groupBy("SUGGESTED_CATEGORY")
    .agg(F.countDistinct("universe_name").alias("universes"), F.count("*").alias("data_items"))
    .orderBy(F.col("data_items").desc())
)

### Second: categorize data entries in WEBI level

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load dictionary lineage and category tables
dict_lineage_raw = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage")
category = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_table_category")
universe_def = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition")

# Clean universe_name and DataSource_Name: strip '_old_udt' suffix and '.unx' extension
dict_lineage = dict_lineage_raw \
    .withColumn("universe_name_clean",
        F.trim(F.regexp_replace(F.col("universe_name"), r"(_old_udt|\.unx)$", ""))
    ) \
    .withColumn("DataSource_Name_clean",
        F.trim(F.regexp_replace(F.col("DataSource_Name"), r"(_old_udt|\.unx)$", ""))
    )

# Effective universe: use universe_name when available, fall back to DataSource_Name.
# Both columns identify the BO universe context — DataSource_Name is the display name
# of the data provider which equals the universe name in most cases.
dict_lineage = dict_lineage \
    .withColumn("effective_universe",
        F.upper(F.trim(F.coalesce(
            F.when(F.trim(F.col("universe_name_clean")) != "", F.col("universe_name_clean")),
            F.col("DataSource_Name_clean")
        )))
    )

# Extract sql_table_extracted and sql_schema_extracted from sql_definition.
# sql_definition patterns: "TABLE.COLUMN", "SCHEMA.TABLE.COLUMN", "func(TABLE.COL)".
# Step 1: extract the first word-before-dot (may be schema or table)
# Step 2: if that word is a known Oracle schema, extract the NEXT word as the real table.
KNOWN_SCHEMAS = ["APPS", "ORABOFP", "ORABUP0", "ORADMART1", "ORFM", "ORFP",
                 "APPLSYS", "GL", "AP", "AR", "ORASTAG1", "ORAFIN"]

dict_lineage = dict_lineage \
    .withColumn("_first_token",
        F.upper(F.trim(F.regexp_extract(F.col("sql_definition"), r"([A-Za-z_][A-Za-z0-9_]*)\.", 1)))
    ) \
    .withColumn("sql_schema_extracted",
        F.when(F.col("_first_token").isin(KNOWN_SCHEMAS), F.col("_first_token"))
        .otherwise(F.lit(None))
    ) \
    .withColumn("sql_table_extracted",
        # If first token was a schema, grab the next word after the dot
        F.when(F.col("sql_schema_extracted").isNotNull(),
            F.upper(F.trim(F.regexp_extract(F.col("sql_definition"),
                r"[A-Za-z_][A-Za-z0-9_]*\.([A-Za-z_][A-Za-z0-9_]*)(?:\.|\s|$)", 1)))
        ).otherwise(F.col("_first_token"))
    ) \
    .drop("_first_token")

# ── LEVEL 0: Direct DataItem_id match via universe_dataentry_definition ────────────
# Most precise level: cell 4 already categorized each (universe, DataItem_id) pair.
# Re-uses that work directly rather than re-deriving from scratch.
dataitem_category = universe_def \
    .filter(F.col("SUGGESTED_CATEGORY").isNotNull()) \
    .select(
        F.upper(F.trim(F.col("universe_name"))).alias("def_universe"),
        F.col("DataItem_id").alias("def_DataItem_id"),
        F.col("SUGGESTED_CATEGORY").alias("cat_level0")
    ).dropDuplicates(["def_universe", "def_DataItem_id"])

# ── PRIMARY LOOKUP: (effective_universe, BO_SQL_TABLE) → SUGGESTED_CATEGORY ───────
# sql_table_extracted from sql_definition matches BO_SQL_TABLE (the BO alias).
# Same compound-join design as cell 4 — precise per (universe, table) combination.
precise_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(
            F.upper(F.trim(F.col("UNIVERSE"))),
            F.upper(F.trim(F.col("BO_SQL_TABLE")))
        ).orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("UNIVERSE"))).alias("cat_universe"),
        F.upper(F.trim(F.col("BO_SQL_TABLE"))).alias("cat_bo_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_primary")
    )

# ── FALLBACK 1: (effective_universe, final_table) → SUGGESTED_CATEGORY ──────────
# Catches schema.table format in sql_definition where extracted part = actual table name.
final_table_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(
            F.upper(F.trim(F.col("UNIVERSE"))),
            F.upper(F.trim(F.col("final_table")))
        ).orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("UNIVERSE"))).alias("fb_universe"),
        F.upper(F.trim(F.col("final_table"))).alias("fb_final_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_fallback")
    )

# ── FALLBACK 2: Cross-universe BO_SQL_TABLE lookup (no universe constraint) ──────
# Same table can appear in multiple universes; its category is source-driven so
# it is stable regardless of which universe references it.
cross_univ_bo_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(F.upper(F.trim(F.col("BO_SQL_TABLE"))))
        .orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("BO_SQL_TABLE"))).alias("cx_bo_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_cross_univ")
    )

# ── FALLBACK 3: Universe name keyword matching (last resort) ─────────────────
univ_keyword_expr = (
    F.when(F.col("effective_universe").rlike(
        "(?i)(FINANC|AP.AR.GL|PAYABL|RECEIVABL|AGING|PRUDENCE|PAYMT|PAYMENT.*RECOV"
        "|INVOICE|LEDGER|BUDGET|P2P|EXCHANGE.RATE|SCHED.2|SPANISH.TAX|EKV|CUMULATIVE.CASES.FIN)"
    ), F.lit("Finance / Accounting"))
    .when(F.col("effective_universe").rlike(
        "(?i)(CLAIM|OVERDUE|COLLECTION|GESTION.RECOUVR|INITIAL.CLAIMS|GKS.CLAIMS)"
    ), F.lit("Claims / Collections"))
    .when(F.col("effective_universe").rlike(
        "(?i)(POLICY|POLIC|UNDERWRITING|MODULE.POLICIES|NPP.CORE|COVER|MEXICO|RECONCILIATION_GKS)"
    ), F.lit("Policy / Underwriting"))
    .when(F.col("effective_universe").rlike(
        "(?i)(CREDIT.LIMIT|BUYER|DEBTOR|SMALL.CREDIT|ALL.CREDIT)"
    ), F.lit("Buyer / Debtor"))
    .when(F.col("effective_universe").rlike(
        "(?i)(ARCADE.OFFER|SALES|COMMERCIAL|LEAD.MANAGEMENT)"
    ), F.lit("Sales / Commercial"))
    .when(F.col("effective_universe").rlike(
        "(?i)(PRICING|PIPELINE_PREMIUM)"
    ), F.lit("Premium Management"))
    .when(F.col("effective_universe").rlike(
        "(?i)(BOND|SURETY|GUARANTEE)"
    ), F.lit("Bonds / Surety"))
    .when(F.col("effective_universe").rlike(
        "(?i)(CUSTOMER.SERVICE|PASSPORT)"
    ), F.lit("Customer Service"))
    .when(F.col("effective_universe").rlike(
        "(?i)(CABRILLO|COMBINED.*CLAIMS.*FINANCE|UPR.DETAILED|UPR.DAC)"
    ), F.lit("Reinsurance / Cabrillo"))
    .when(F.col("effective_universe").rlike(
        "(?i)(MEDIUM.TERM|SINGLE.RISK)"
    ), F.lit("Medium-Term / Special Products"))
    .when(F.col("effective_universe").rlike(
        "(?i)(USER.SECURITY|WORKFLOW|WFR.TOOLS|GROUP.HR)"
    ), F.lit("Workflow / User Management"))
    .when(F.col("effective_universe").rlike(
        "(?i)(AM.TOOL|ASSET.MGMT)"
    ), F.lit("Asset Management / Monitoring"))
    .when(F.col("effective_universe").rlike(
        "(?i)(REFERENCE.DATA|ORG.COMM|ORGANISATION|TPE|ABRP|REF_ISO|BELGIUM.CUST)"
    ), F.lit("Organisation / Reference Data"))
    .when(F.col("effective_universe").rlike(
        "(?i)(^WITS|QUALITY.CENTER|KPI)"
    ), F.lit("WITS / IT Service Management"))
    .when(F.col("effective_universe").rlike(
        "(?i)(FHSQL|SQL_INVENTORY|DALLAS|BBT_DWH|PVDATAMART|PVPORT|OID.PRODUCTION)"
    ), F.lit("Data Warehouse / Dimensions"))
)

# --- Level 5: Manual mapping for known unmatched names ---
manual_mapping = {
    "COMBINED AP,AR & GL UNIVERSE": "Finance / Accounting",
    "APARGL11": "Finance / Accounting",
    "FINANCIAL TRANSACTIONS": "Finance / Accounting",
    "FTTRANS": "Finance / Accounting",
    "PRUDENCE PROD (ORFP)": "Finance / Accounting",
    "PRUDENCE (ORFP) PROD": "Finance / Accounting",
    "INVOICE TYPES": "Finance / Accounting",
    "PAYM'T _ RECOVERY FIN": "Finance / Accounting",
    "NEW_AP_DATA": "Finance / Accounting",
    "NEW_AR_DATA": "Finance / Accounting",
    "NEW_GL_DATA": "Finance / Accounting",
    "NEW_GL_DATA_EURO": "Finance / Accounting",
    "NEW_AR_DATA_EURO": "Finance / Accounting",
    "CONTANTS - LEDGER": "Finance / Accounting",
    "CONTANTS - LEGAL ENTITY": "Finance / Accounting",
    "CONTANTS - ORGANIZATION": "Finance / Accounting",
    "GL1": "Finance / Accounting",
    "ACCOUNTS": "Finance / Accounting",
    "UNDERWRITING MONTHLY BSB": "Policy / Underwriting",
    "POLICY MONTHLY BSB": "Policy / Underwriting",
    "POLICY  MONTHLY BSB": "Policy / Underwriting",
    "COMBINED PAYM'T & RECOVERY": "Claims / Collections",
    "ORGANISATIONS": "Organisation / Reference Data",
    "ORG ID": "Organisation / Reference Data",
    "REF TABLE": "Organisation / Reference Data",
    "COUNTRY": "Organisation / Reference Data",
    "DWH PRODUCTION 1": "Data Warehouse / Dimensions",
    "PVDATAMART ATRADIUS": "Data Warehouse / Dimensions",
    "DALLAS": "Data Warehouse / Dimensions",
    "SYMP_RED_HOT": "Risk / Credit Assessment",
    "FHSQL046": "ERP / System Config",
    "FHSQL050": "ERP / System Config",
    "FHSQL050 (2)": "ERP / System Config",
    # --- Additional mappings for unmatched DataSource_Names (universe_name is NULL) ---
    # Finance / Accounting
    "SQLDBUK\\ATRCYCDB_GFO_BASWAREINVOICING": "Finance / Accounting",
    "RECONCIL": "Finance / Accounting",
    "CUMULATIVE CASES FIN_POL YR": "Finance / Accounting",
    "SPANISH TAX": "Finance / Accounting",
    "FC BMR INPUT": "Finance / Accounting",
    "GESTION CAUTION": "Finance / Accounting",
    "COMBINED AP,AR & GL WITH XLA": "Finance / Accounting",
    # Policy / Underwriting
    "CRLICOMB": "Policy / Underwriting",
    "POLICY  BSB": "Policy / Underwriting",
    "MODULE POLICIES MONTHLY BSB": "Policy / Underwriting",
    "HOLDCOVER_GSI_JSW": "Policy / Underwriting",
    "GOVT SCHEME CYCLE DATES": "Policy / Underwriting",
    # Organisation / Reference Data
    "SHIPMENT": "Organisation / Reference Data",
    "ORG ID & DESCRIPTION": "Organisation / Reference Data",
    "DATE LOOKUP": "Organisation / Reference Data",
    "CYCLE DATES": "Organisation / Reference Data",
    # Claims / Collections
    "OVERDUES CLAIMS MONTHLY BSB": "Claims / Collections",
    "MANUAL IMPORT CUSTOMERS GETPAID": "Claims / Collections",
    "CLA_ID_ML_TREE_MAY_2023_MAR_2024": "Claims / Collections",
    # Buyer / Debtor
    "LIST_BUYER_CSC": "Buyer / Debtor",
    "LIMITS BNP BE CRM DIRECT": "Buyer / Debtor",
    "TEST_BUYER_RATING": "Buyer / Debtor",
    # Data Warehouse / Dimensions
    "PRODUCTION REPOS UNIVERSE": "Data Warehouse / Dimensions",
    # WITS / IT Service Management
    "REMEDY HELPDESK & LEGACY": "WITS / IT Service Management",
    # Workflow / User Management
    "BI PLATFORM AUDIT ANALYSIS": "Workflow / User Management",
    "USER IDS FOR UK AND IRELAND COMMERCIAL": "Workflow / User Management",
    "UID": "Workflow / User Management",
    # Other / Unclassified
    "EFASHION": "Other / Unclassified",
    "DASHBOARD_CONSTANTS": "Other / Unclassified",
    "SRS ESTERE V2": "Other / Unclassified",
    "SYMQ MIX": "Other / Unclassified",
}

manual_mapping_expr_univ = F.lit(None).cast("string")
for name, cat in manual_mapping.items():
    manual_mapping_expr_univ = F.when(
        F.upper(F.trim(F.col("universe_name_clean"))) == name, F.lit(cat)
    ).otherwise(manual_mapping_expr_univ)

manual_mapping_expr_ds = F.lit(None).cast("string")
for name, cat in manual_mapping.items():
    manual_mapping_expr_ds = F.when(
        F.upper(F.trim(F.col("DataSource_Name_clean"))) == name, F.lit(cat)
    ).otherwise(manual_mapping_expr_ds)

# --- Multi-level join and coalesce ---
dict_categorized = dict_lineage \
    .join(dataitem_category,
        (F.col("effective_universe") == F.col("def_universe")) &
        (F.col("DataItem_id") == F.col("def_DataItem_id")),
        "left") \
    .join(precise_lookup,
        (F.col("effective_universe") == F.col("cat_universe")) &
        (F.col("sql_table_extracted") == F.col("cat_bo_table")),
        "left") \
    .join(final_table_lookup,
        (F.col("effective_universe") == F.col("fb_universe")) &
        (F.col("sql_table_extracted") == F.col("fb_final_table")),
        "left") \
    .join(cross_univ_bo_lookup,
        F.col("sql_table_extracted") == F.col("cx_bo_table"),
        "left") \
    .withColumn("manual_cat_univ", manual_mapping_expr_univ) \
    .withColumn("manual_cat_ds", manual_mapping_expr_ds) \
    .withColumn("univ_keyword_cat", univ_keyword_expr) \
    .withColumn("UNIVERSE_PRIMARY_CATEGORY",
        F.coalesce(
            F.col("cat_level0"),      # Level 0:  Direct (universe, DataItem_id) match from cell 4
            F.col("cat_primary"),     # Primary:  (effective_universe, BO_SQL_TABLE) compound join
            F.col("cat_fallback"),    # Fallback 1: (effective_universe, final_table) compound join
            # Fallback 2: ORFM/ORFP schema in sql_definition → Finance / Accounting
            F.when(F.col("sql_schema_extracted").isin("ORFM", "ORFP"),
                   F.lit("Finance / Accounting")),
            F.col("cat_cross_univ"),  # Fallback 3: cross-universe BO_SQL_TABLE lookup
            F.col("manual_cat_univ"), # Fallback 4: manual mapping on universe_name
            F.col("manual_cat_ds"),   # Fallback 5: manual mapping on DataSource_Name
            F.col("univ_keyword_cat") # Fallback 6: universe name keywords (last resort)
        )
    ) \
    .select(
        dict_lineage["Document_Id"],
        dict_lineage["Document_name"],
        dict_lineage["Data_Provider_ID"],
        dict_lineage["datafield_id"],
        dict_lineage["datafield_name"],
        dict_lineage["datafield_qualification"],
        dict_lineage["datafield_datatype"],
        dict_lineage["datafield_name_universe"],
        dict_lineage["datafield_datasourceObjectId"],
        dict_lineage["DataItem_id"],
        dict_lineage["DataSource_Name"],
        dict_lineage["universe_name"],
        dict_lineage["datafield_description"],
        dict_lineage["datafield_datasourcepath"],
        dict_lineage["DataItem_path"],
        dict_lineage["sql_definition"],
        F.col("UNIVERSE_PRIMARY_CATEGORY")
    )

# Write categorized result back
dict_categorized.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage")

# Summary stats
total = dict_categorized.count()
distinct_docs = dict_categorized.select("Document_Id").distinct().count()
distinct_items = dict_categorized.select("Document_Id", "datafield_id").distinct().count()
matched = dict_categorized.filter(F.col("UNIVERSE_PRIMARY_CATEGORY").isNotNull()).count()

print(f"Total rows: {total:,}")
print(f"Distinct Documents: {distinct_docs:,}")
print(f"Distinct Dataitems (Doc + datafield_id): {distinct_items:,}")
print(f"\nMatched (has category): {matched:,} ({100*matched/total:.1f}%)")
print(f"Unmatched (NULL):       {total-matched:,} ({100*(total-matched)/total:.1f}%)")
print(f"\n=== CATEGORY DISTRIBUTION ACROSS DATAITEMS ===")

display(
    dict_categorized
    .groupBy("UNIVERSE_PRIMARY_CATEGORY")
    .agg(
        F.countDistinct("Document_Id").alias("documents"),
        F.countDistinct("Document_Id", "datafield_id").alias("unique_dataitems"),
        F.count("*").alias("total_rows")
    )
    .orderBy(F.col("unique_dataitems").desc())
)

### Third: Categorize Variables 
with looping variable-variable reference

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load variable lineage, categorized dictionary lineage, and category lookups
lineage_raw = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")
dict_lineage_raw = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage")
category = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_table_category")

# Clean Universe_Name / DataSource_Name the same way as cell 6
lineage = lineage_raw \
    .withColumn("Universe_Name_clean",
        F.trim(F.regexp_replace(F.col("Universe_Name"), r"(_old_udt|\.unx)$", ""))
    ) \
    .withColumn("DataSource_Name_clean",
        F.trim(F.regexp_replace(F.col("DataSource_Name"), r"(_old_udt|\.unx)$", ""))
    ) \
    .withColumn("effective_universe",
        F.upper(F.trim(F.coalesce(
            F.when(F.trim(F.col("Universe_Name_clean")) != "", F.col("Universe_Name_clean")),
            F.col("DataSource_Name_clean")
        )))
    ) \
    .withColumn("universe_name_key", F.upper(F.trim(F.col("Universe_Name_clean")))) \
    .withColumn("datasource_name_key", F.upper(F.trim(F.col("DataSource_Name_clean")))) \
    .withColumn("datasource_path_key", F.lower(F.trim(F.col("datafield_datasourcePath")))) \
    .withColumn("datafield_name_key", F.upper(F.trim(F.col("datafield_name"))))

dict_lineage = dict_lineage_raw \
    .withColumn("universe_name_clean",
        F.trim(F.regexp_replace(F.col("universe_name"), r"(_old_udt|\.unx)$", ""))
    ) \
    .withColumn("DataSource_Name_clean",
        F.trim(F.regexp_replace(F.col("DataSource_Name"), r"(_old_udt|\.unx)$", ""))
    ) \
    .withColumn("effective_universe",
        F.upper(F.trim(F.coalesce(
            F.when(F.trim(F.col("universe_name_clean")) != "", F.col("universe_name_clean")),
            F.col("DataSource_Name_clean")
        )))
    ) \
    .withColumn("universe_name_key", F.upper(F.trim(F.col("universe_name_clean")))) \
    .withColumn("datasource_name_key", F.upper(F.trim(F.col("DataSource_Name_clean")))) \
    .withColumn("datasource_path_key", F.lower(F.trim(F.col("datafield_datasourcepath")))) \
    .withColumn("datafield_name_key", F.upper(F.trim(F.col("datafield_name"))))

# Extract the token immediately before the last dot in sql_definition.
# This handles both TABLE.COLUMN and SCHEMA.TABLE.COLUMN patterns.
KNOWN_SCHEMAS = ["APPS", "ORABOFP", "ORABUP0", "ORADMART1", "ORFM", "ORFP",
                 "APPLSYS", "GL", "AP", "AR", "ORASTAG1", "ORAFIN"]

lineage = lineage \
    .withColumn("_prior_token",
        F.upper(F.trim(F.regexp_extract(
            F.col("sql_definition"),
            r"([A-Za-z_][A-Za-z0-9_]*)\.([A-Za-z_][A-Za-z0-9_]*)(?=\.[^.]*$)",
            1
        )))
    ) \
    .withColumn("sql_table_extracted",
        F.upper(F.trim(F.regexp_extract(
            F.col("sql_definition"),
            r"([A-Za-z_][A-Za-z0-9_]*)(?=\.[^.]*$)",
            1
        )))
    ) \
    .withColumn("sql_schema_extracted",
        F.when(F.col("_prior_token").isin(KNOWN_SCHEMAS), F.col("_prior_token"))
         .otherwise(F.lit(None))
    ) \
    .drop("_prior_token")

# ── PRIMARY: (Document_Id, datafield_id) → categorized dictionary lineage ─────────
dict_cat_primary = dict_lineage \
    .filter(F.col("UNIVERSE_PRIMARY_CATEGORY").isNotNull()) \
    .select(
        F.col("Document_Id").alias("d_doc_id"),
        F.col("datafield_id").alias("d_datafield_id"),
        F.col("UNIVERSE_PRIMARY_CATEGORY").alias("cat_primary")
    ) \
    .dropDuplicates(["d_doc_id", "d_datafield_id"])

# ── FALLBACK 1: (Universe_Name, datafield_datasourceObjectId) ─────────────────────
dict_cat_fb1 = dict_lineage \
    .filter(
        F.col("UNIVERSE_PRIMARY_CATEGORY").isNotNull() &
        F.col("universe_name_key").isNotNull() &
        (F.col("universe_name_key") != "") &
        F.col("datafield_datasourceObjectId").isNotNull()
    ) \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("universe_name_key", "datafield_datasourceObjectId")
              .orderBy(F.col("UNIVERSE_PRIMARY_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.col("universe_name_key").alias("fb1_universe"),
        F.col("datafield_datasourceObjectId").alias("fb1_obj_id"),
        F.col("UNIVERSE_PRIMARY_CATEGORY").alias("cat_fallback1")
    )

# ── FALLBACK 2: (Universe_Name, datafield_datasourcePath, datafield_name) ─────────
dict_cat_fb2 = dict_lineage \
    .filter(
        F.col("UNIVERSE_PRIMARY_CATEGORY").isNotNull() &
        F.col("universe_name_key").isNotNull() &
        (F.col("universe_name_key") != "") &
        F.col("datasource_path_key").isNotNull() &
        (F.col("datasource_path_key") != "") &
        F.col("datafield_name_key").isNotNull() &
        (F.col("datafield_name_key") != "")
    ) \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("universe_name_key", "datasource_path_key", "datafield_name_key")
              .orderBy(F.col("UNIVERSE_PRIMARY_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.col("universe_name_key").alias("fb2_universe"),
        F.col("datasource_path_key").alias("fb2_path"),
        F.col("datafield_name_key").alias("fb2_name"),
        F.col("UNIVERSE_PRIMARY_CATEGORY").alias("cat_fallback2")
    )

# ── FALLBACK 3: (Universe_Name, table before last dot) → BO_SQL_TABLE ─────────────
precise_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(
            F.upper(F.trim(F.col("UNIVERSE"))),
            F.upper(F.trim(F.col("BO_SQL_TABLE")))
        ).orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("UNIVERSE"))).alias("cat_universe"),
        F.upper(F.trim(F.col("BO_SQL_TABLE"))).alias("cat_bo_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_precise")
    )

# ── FALLBACK 4: (Universe_Name, table before last dot) → final_table ──────────────
final_table_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(
            F.upper(F.trim(F.col("UNIVERSE"))),
            F.upper(F.trim(F.col("final_table")))
        ).orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("UNIVERSE"))).alias("fb4_universe"),
        F.upper(F.trim(F.col("final_table"))).alias("fb4_final_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_fallback4")
    )

# ── FALLBACK 5/6 continued from cell 6 logic ───────────────────────────────────────
cross_univ_bo_lookup = category \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy(F.upper(F.trim(F.col("BO_SQL_TABLE"))))
              .orderBy(F.col("SUGGESTED_CATEGORY"))
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.upper(F.trim(F.col("BO_SQL_TABLE"))).alias("cx_bo_table"),
        F.col("SUGGESTED_CATEGORY").alias("cat_cross_univ")
    )

# --- Manual mapping for known unmatched names (same expanded mapping as cell 6) ---
manual_mapping = {
    "COMBINED AP,AR & GL UNIVERSE": "Finance / Accounting",
    "APARGL11": "Finance / Accounting",
    "FINANCIAL TRANSACTIONS": "Finance / Accounting",
    "FTTRANS": "Finance / Accounting",
    "PRUDENCE PROD (ORFP)": "Finance / Accounting",
    "PRUDENCE (ORFP) PROD": "Finance / Accounting",
    "INVOICE TYPES": "Finance / Accounting",
    "PAYM'T _ RECOVERY FIN": "Finance / Accounting",
    "NEW_AP_DATA": "Finance / Accounting",
    "NEW_AR_DATA": "Finance / Accounting",
    "NEW_GL_DATA": "Finance / Accounting",
    "NEW_GL_DATA_EURO": "Finance / Accounting",
    "NEW_AR_DATA_EURO": "Finance / Accounting",
    "CONTANTS - LEDGER": "Finance / Accounting",
    "CONTANTS - LEGAL ENTITY": "Finance / Accounting",
    "CONTANTS - ORGANIZATION": "Finance / Accounting",
    "GL1": "Finance / Accounting",
    "ACCOUNTS": "Finance / Accounting",
    "UNDERWRITING MONTHLY BSB": "Policy / Underwriting",
    "POLICY MONTHLY BSB": "Policy / Underwriting",
    "POLICY  MONTHLY BSB": "Policy / Underwriting",
    "COMBINED PAYM'T & RECOVERY": "Claims / Collections",
    "ORGANISATIONS": "Organisation / Reference Data",
    "ORG ID": "Organisation / Reference Data",
    "REF TABLE": "Organisation / Reference Data",
    "COUNTRY": "Organisation / Reference Data",
    "DWH PRODUCTION 1": "Data Warehouse / Dimensions",
    "PVDATAMART ATRADIUS": "Data Warehouse / Dimensions",
    "DALLAS": "Data Warehouse / Dimensions",
    "SYMP_RED_HOT": "Risk / Credit Assessment",
    "FHSQL046": "ERP / System Config",
    "FHSQL050": "ERP / System Config",
    "FHSQL050 (2)": "ERP / System Config",
    "SQLDBUK\\ATRCYCDB_GFO_BASWAREINVOICING": "Finance / Accounting",
    "RECONCIL": "Finance / Accounting",
    "CUMULATIVE CASES FIN_POL YR": "Finance / Accounting",
    "SPANISH TAX": "Finance / Accounting",
    "FC BMR INPUT": "Finance / Accounting",
    "GESTION CAUTION": "Finance / Accounting",
    "COMBINED AP,AR & GL WITH XLA": "Finance / Accounting",
    "CRLICOMB": "Policy / Underwriting",
    "POLICY  BSB": "Policy / Underwriting",
    "MODULE POLICIES MONTHLY BSB": "Policy / Underwriting",
    "HOLDCOVER_GSI_JSW": "Policy / Underwriting",
    "GOVT SCHEME CYCLE DATES": "Policy / Underwriting",
    "SHIPMENT": "Organisation / Reference Data",
    "ORG ID & DESCRIPTION": "Organisation / Reference Data",
    "DATE LOOKUP": "Organisation / Reference Data",
    "CYCLE DATES": "Organisation / Reference Data",
    "OVERDUES CLAIMS MONTHLY BSB": "Claims / Collections",
    "MANUAL IMPORT CUSTOMERS GETPAID": "Claims / Collections",
    "CLA_ID_ML_TREE_MAY_2023_MAR_2024": "Claims / Collections",
    "LIST_BUYER_CSC": "Buyer / Debtor",
    "LIMITS BNP BE CRM DIRECT": "Buyer / Debtor",
    "TEST_BUYER_RATING": "Buyer / Debtor",
    "PRODUCTION REPOS UNIVERSE": "Data Warehouse / Dimensions",
    "REMEDY HELPDESK & LEGACY": "WITS / IT Service Management",
    "BI PLATFORM AUDIT ANALYSIS": "Workflow / User Management",
    "USER IDS FOR UK AND IRELAND COMMERCIAL": "Workflow / User Management",
    "UID": "Workflow / User Management",
    "EFASHION": "Other / Unclassified",
    "DASHBOARD_CONSTANTS": "Other / Unclassified",
    "SRS ESTERE V2": "Other / Unclassified",
    "SYMQ MIX": "Other / Unclassified",
}

manual_mapping_expr_univ = F.lit(None).cast("string")
for name, cat in manual_mapping.items():
    manual_mapping_expr_univ = F.when(
        F.col("universe_name_key") == name, F.lit(cat)
    ).otherwise(manual_mapping_expr_univ)

manual_mapping_expr_ds = F.lit(None).cast("string")
for name, cat in manual_mapping.items():
    manual_mapping_expr_ds = F.when(
        F.col("datasource_name_key") == name, F.lit(cat)
    ).otherwise(manual_mapping_expr_ds)

# Last-resort keyword matching from cell 6
univ_keyword_expr = (
    F.when(F.col("effective_universe").rlike(
        "(?i)(FINANC|AP.AR.GL|PAYABL|RECEIVABL|AGING|PRUDENCE|PAYMT|PAYMENT.*RECOV"
        "|INVOICE|LEDGER|BUDGET|P2P|EXCHANGE.RATE|SCHED.2|SPANISH.TAX|EKV|CUMULATIVE.CASES.FIN)"
    ), F.lit("Finance / Accounting"))
    .when(F.col("effective_universe").rlike(
        "(?i)(CLAIM|OVERDUE|COLLECTION|GESTION.RECOUVR|INITIAL.CLAIMS|GKS.CLAIMS)"
    ), F.lit("Claims / Collections"))
    .when(F.col("effective_universe").rlike(
        "(?i)(POLICY|POLIC|UNDERWRITING|MODULE.POLICIES|NPP.CORE|COVER|MEXICO|RECONCILIATION_GKS)"
    ), F.lit("Policy / Underwriting"))
    .when(F.col("effective_universe").rlike(
        "(?i)(CREDIT.LIMIT|BUYER|DEBTOR|SMALL.CREDIT|ALL.CREDIT)"
    ), F.lit("Buyer / Debtor"))
    .when(F.col("effective_universe").rlike(
        "(?i)(ARCADE.OFFER|SALES|COMMERCIAL|LEAD.MANAGEMENT)"
    ), F.lit("Sales / Commercial"))
    .when(F.col("effective_universe").rlike(
        "(?i)(PRICING|PIPELINE_PREMIUM)"
    ), F.lit("Premium Management"))
    .when(F.col("effective_universe").rlike(
        "(?i)(BOND|SURETY|GUARANTEE)"
    ), F.lit("Bonds / Surety"))
    .when(F.col("effective_universe").rlike(
        "(?i)(CUSTOMER.SERVICE|PASSPORT)"
    ), F.lit("Customer Service"))
    .when(F.col("effective_universe").rlike(
        "(?i)(CABRILLO|COMBINED.*CLAIMS.*FINANCE|UPR.DETAILED|UPR.DAC)"
    ), F.lit("Reinsurance / Cabrillo"))
    .when(F.col("effective_universe").rlike(
        "(?i)(MEDIUM.TERM|SINGLE.RISK)"
    ), F.lit("Medium-Term / Special Products"))
    .when(F.col("effective_universe").rlike(
        "(?i)(USER.SECURITY|WORKFLOW|WFR.TOOLS|GROUP.HR)"
    ), F.lit("Workflow / User Management"))
    .when(F.col("effective_universe").rlike(
        "(?i)(AM.TOOL|ASSET.MGMT)"
    ), F.lit("Asset Management / Monitoring"))
    .when(F.col("effective_universe").rlike(
        "(?i)(REFERENCE.DATA|ORG.COMM|ORGANISATION|TPE|ABRP|REF_ISO|BELGIUM.CUST)"
    ), F.lit("Organisation / Reference Data"))
    .when(F.col("effective_universe").rlike(
        "(?i)(^WITS|QUALITY.CENTER|KPI)"
    ), F.lit("WITS / IT Service Management"))
    .when(F.col("effective_universe").rlike(
        "(?i)(FHSQL|SQL_INVENTORY|DALLAS|BBT_DWH|PVDATAMART|PVPORT|OID.PRODUCTION)"
    ), F.lit("Data Warehouse / Dimensions"))
)

# --- Multi-level join and coalesce ---
variable_categorized = lineage \
    .join(dict_cat_primary,
        (F.col("Document_Id") == F.col("d_doc_id")) &
        (F.col("datafield_id") == F.col("d_datafield_id")),
        "left") \
    .join(dict_cat_fb1,
        (F.col("universe_name_key") == F.col("fb1_universe")) &
        (F.col("datafield_datasourceObjectId") == F.col("fb1_obj_id")),
        "left") \
    .join(dict_cat_fb2,
        (F.col("universe_name_key") == F.col("fb2_universe")) &
        (F.col("datasource_path_key") == F.col("fb2_path")) &
        (F.col("datafield_name_key") == F.col("fb2_name")),
        "left") \
    .join(precise_lookup,
        (F.col("universe_name_key") == F.col("cat_universe")) &
        (F.col("sql_table_extracted") == F.col("cat_bo_table")),
        "left") \
    .join(final_table_lookup,
        (F.col("universe_name_key") == F.col("fb4_universe")) &
        (F.col("sql_table_extracted") == F.col("fb4_final_table")),
        "left") \
    .join(cross_univ_bo_lookup,
        F.col("sql_table_extracted") == F.col("cx_bo_table"),
        "left") \
    .withColumn("manual_cat_univ", manual_mapping_expr_univ) \
    .withColumn("manual_cat_ds", manual_mapping_expr_ds) \
    .withColumn("univ_keyword_cat", univ_keyword_expr) \
    .withColumn("UNIVERSE_PRIMARY_CATEGORY",
        F.coalesce(
            F.col("cat_primary"),
            F.col("cat_fallback1"),
            F.col("cat_fallback2"),
            F.col("cat_precise"),
            F.col("cat_fallback4"),
            F.when(F.col("sql_schema_extracted").isin("ORFM", "ORFP"),
                   F.lit("Finance / Accounting")),
            F.col("cat_cross_univ"),
            F.col("manual_cat_univ"),
            F.col("manual_cat_ds"),
            F.col("univ_keyword_cat"),
            F.lit("Other / Unclassified")  # Final fallback
        )
    ) \
    .select(
        F.col("Document_Id"),
        F.col("variable_id"),
        F.col("variable_name"),
        F.col("variable_datatype"),
        F.col("variable_qualification"),
        F.col("variable_definition"),
        F.col("extracted_datafield"),
        F.col("provider_qualifier"),
        F.col("Data_Provider_ID"),
        F.col("datafield_id"),
        F.col("datafield_name"),
        F.col("datafield_qualification"),
        F.col("DataSource_Type"),
        F.col("DataSource_Name_clean").alias("DataSource_Name"),
        F.col("datafield_datasourceObjectId"),
        F.col("Universe_ID"),
        F.col("Universe_Name_clean").alias("Universe_Name"),
        F.col("webi_all_sql_queries"),
        F.col("datafield_datasourcePath"),
        F.col("datafield_description"),
        F.col("sql_definition"),
        F.col("UNIVERSE_PRIMARY_CATEGORY")
    )

# Summary stats
total_vars = variable_categorized.count()
distinct_docs = variable_categorized.select("Document_Id").distinct().count()
distinct_vars = variable_categorized.select("Document_Id", "variable_id").distinct().count()
matched = variable_categorized.filter(F.col("UNIVERSE_PRIMARY_CATEGORY").isNotNull()).count()

print(f"Total rows: {total_vars:,}")
print(f"Distinct Documents: {distinct_docs:,}")
print(f"Distinct Variables (Doc + Var): {distinct_vars:,}")
print(f"\nMatched (has category): {matched:,} ({100*matched/total_vars:.1f}%)")
print(f"Unmatched (NULL):       {total_vars - matched:,} ({100*(total_vars-matched)/total_vars:.1f}%)")
print(f"\n=== CATEGORY DISTRIBUTION ACROSS VARIABLES ===")

display(
    variable_categorized
    .groupBy("UNIVERSE_PRIMARY_CATEGORY")
    .agg(
        F.countDistinct("Document_Id").alias("documents"),
        F.countDistinct("Document_Id", "variable_id").alias("unique_variables"),
        F.count("*").alias("total_rows")
    )
    .orderBy(F.col("unique_variables").desc())
)

In [0]:
from pyspark.sql import functions as F

# Write variable_categorized (computed in cell 8) to the target table.
# No join logic here — cell 8 owns all categorization.
target = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage"

variable_categorized.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target)

result = spark.table(target)
total   = result.count()
matched = result.filter(F.col("UNIVERSE_PRIMARY_CATEGORY").isNotNull()).count()

print(f"\u2713 Written {total:,} rows to {target}")
print(f"  Category filled: {matched:,} ({100*matched/total:.1f}%)")
print(f"  Category NULL:   {total-matched:,} ({100*(total-matched)/total:.1f}%)")

In [0]:
from pyspark.sql import functions as F

# Load both categorized tables
dict_df = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage")
vars_df = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")

# --- Dictionary entries ---
dict_entries = dict_df.select(
    F.lit("DICTIONARY").alias("entry_type"),
    F.col("Document_Id"),
    F.col("Document_name"),
    F.col("Data_Provider_ID"),
    # Entry identity
    F.col("datafield_id").alias("entry_id"),
    F.col("datafield_name").alias("entry_name"),
    F.col("datafield_datatype").alias("entry_datatype"),
    F.col("datafield_qualification").alias("entry_qualification"),
    # Variable-specific (NULL for dictionary)
    F.lit(None).cast("string").alias("variable_definition"),
    F.lit(None).cast("string").alias("extracted_datafield"),
    F.lit(None).cast("string").alias("provider_qualifier"),
    # Reference: dictionary items ARE the base, no reference needed
    F.lit(None).cast("string").alias("referenced_datafield_id"),
    F.lit(None).cast("string").alias("referenced_datafield_name"),
    # Universe / DataSource context
    F.col("DataSource_Name"),
    F.col("universe_name").alias("Universe_Name"),
    F.col("datafield_datasourceObjectId"),
    F.col("DataItem_id"),
    F.col("datafield_name_universe"),
    F.col("datafield_datasourcepath"),
    F.col("DataItem_path"),
    F.col("datafield_description"),
    F.col("sql_definition"),
    # Category
    F.col("UNIVERSE_PRIMARY_CATEGORY")
)

# --- Variable entries ---
vars_entries = vars_df.select(
    F.lit("VARIABLE").alias("entry_type"),
    F.col("Document_Id"),
    F.lit(None).cast("string").alias("Document_name"),  # not in variables table
    F.col("Data_Provider_ID"),
    # Entry identity
    F.col("variable_id").alias("entry_id"),
    F.col("variable_name").alias("entry_name"),
    F.col("variable_datatype").alias("entry_datatype"),
    F.col("variable_qualification").alias("entry_qualification"),
    # Variable-specific
    F.col("variable_definition"),
    F.col("extracted_datafield"),
    F.col("provider_qualifier"),
    # Reference: which dictionary datafield this variable uses
    F.col("datafield_id").alias("referenced_datafield_id"),
    F.col("datafield_name").alias("referenced_datafield_name"),
    # Universe / DataSource context
    F.col("DataSource_Name"),
    F.col("Universe_Name"),
    F.col("datafield_datasourceObjectId"),
    F.lit(None).cast("string").alias("DataItem_id"),  # not in variables table
    F.lit(None).cast("string").alias("datafield_name_universe"),
    F.col("datafield_datasourcePath").alias("datafield_datasourcepath"),
    F.lit(None).cast("string").alias("DataItem_path"),
    F.col("datafield_description"),
    F.col("sql_definition"),
    # Category
    F.col("UNIVERSE_PRIMARY_CATEGORY")
)

# --- Union both ---
webi_data_entries = dict_entries.unionByName(vars_entries)

# Write to target
target = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_data_entries"
webi_data_entries.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)

# Summary
total = webi_data_entries.count()
dict_count = webi_data_entries.filter(F.col("entry_type") == "DICTIONARY").count()
vars_count = webi_data_entries.filter(F.col("entry_type") == "VARIABLE").count()
docs = webi_data_entries.select("Document_Id").distinct().count()
matched = webi_data_entries.filter(F.col("UNIVERSE_PRIMARY_CATEGORY").isNotNull()).count()

print(f"=== webi_data_entries written to {target} ===")
print(f"Total rows:        {total:,}")
print(f"  DICTIONARY:      {dict_count:,}")
print(f"  VARIABLE:        {vars_count:,}")
print(f"Distinct Documents:{docs:,}")
print(f"Category coverage: {matched:,} ({100*matched/total:.1f}%)")
print(f"\n=== CATEGORY DISTRIBUTION ===")

display(
    webi_data_entries
    .groupBy("UNIVERSE_PRIMARY_CATEGORY", "entry_type")
    .agg(
        F.countDistinct("Document_Id").alias("documents"),
        F.count("*").alias("entries")
    )
    .orderBy("UNIVERSE_PRIMARY_CATEGORY", "entry_type")
)

In [0]:
from pyspark.sql import functions as F

T = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_data_entries"
df = spark.table(T)

# ── 1. High-level counts ──────────────────────────────────────────────────────
total     = df.count()
dict_rows = df.filter(F.col("entry_type") == "DICTIONARY").count()
var_rows  = df.filter(F.col("entry_type") == "VARIABLE").count()

print("=" * 65)
print(f"Total rows     : {total:,}")
print(f"  DICTIONARY   : {dict_rows:,}")
print(f"  VARIABLE     : {var_rows:,}")

# ── 2. TRUE uniqueness check: (Document_Id, entry_type, entry_id, dict_item) ──
# This is the correct key: one row = one (entry → dict_item) mapping.
# A variable referencing N distinct dict items legitimately produces N rows.
extended_key = ["Document_Id", "entry_type", "entry_id",
                F.coalesce(F.col("referenced_datafield_id"), F.lit("__NULL__")).alias("_dict_key")]
extended_distinct = df.select(
    F.col("Document_Id"), F.col("entry_type"), F.col("entry_id"),
    F.coalesce(F.col("referenced_datafield_id"), F.lit("__NULL__")).alias("_dict_key")
).distinct().count()
true_duplicates = total - extended_distinct

print("\n" + "=" * 65)
print(f"[CHECK 1] True duplicates on (Doc, type, entry_id, dict_item)")
print(f"  Extended key distinct : {extended_distinct:,}")
print(f"  True duplicate rows   : {true_duplicates:,}",
      "✓" if true_duplicates == 0 else "✗ ACTION NEEDED")

if true_duplicates > 0:
    true_dup_keys = (
        df.groupBy(
            "Document_Id", "entry_type", "entry_id",
            F.coalesce(F.col("referenced_datafield_id"), F.lit("__NULL__")).alias("dict_item")
        )
        .agg(F.count("*").alias("row_count"))
        .filter(F.col("row_count") > 1)
        .orderBy(F.col("row_count").desc())
    )
    print("\nTop true duplicate keys (showing 20):")
    display(true_dup_keys.limit(20))

# ── 3. Informational: multi-dict variables ────────────────────────────────────
# Variables with >1 row have multiple DISTINCT dict items — this is expected.
# Each row = a different data dictionary field the variable ultimately traces to.
var_dict_counts = (
    df.filter(F.col("entry_type") == "VARIABLE")
      .groupBy("Document_Id", "entry_id")
      .agg(
          F.count("*").alias("total_rows"),
          F.countDistinct(
              F.coalesce(F.col("referenced_datafield_id"), F.lit("__NULL__"))
          ).alias("distinct_dict_items")
      )
)

dist_summary = (
    var_dict_counts
    .groupBy("distinct_dict_items")
    .agg(F.count("*").alias("variable_count"))
    .orderBy("distinct_dict_items")
)

distinct_var_entries = var_dict_counts.count()
print("\n" + "=" * 65)
print(f"[INFO] Variable entries (Doc + variable_id combinations): {distinct_var_entries:,}")
print(f"       Variable rows total                               : {var_rows:,}")
print(f"       Extra rows = variables with multiple dict refs    : {var_rows - distinct_var_entries:,}  (expected)")
print("\nDistribution — distinct dict items mapped per variable:")
display(dist_summary)

# ── 4. Variable → Variable chains ────────────────────────────────────────────
# A VARIABLE whose referenced_datafield_id matches another VARIABLE's entry_id
# in the same document is an unresolved var→var link (should be 0 after Step 6).
var_ids = (
    df.filter(F.col("entry_type") == "VARIABLE")
      .select(F.col("Document_Id").alias("ref_doc"),
              F.col("entry_id").alias("ref_entry_id"))
      .distinct()
)
var_to_var = (
    df.filter(
        (F.col("entry_type") == "VARIABLE") &
        F.col("referenced_datafield_id").isNotNull()
    )
    .join(
        var_ids,
        (df["Document_Id"] == var_ids["ref_doc"]) &
        (df["referenced_datafield_id"] == var_ids["ref_entry_id"]),
        "inner"
    )
    .select(df["Document_Id"],
            df["entry_id"].alias("variable_id"),
            df["entry_name"].alias("variable_name"),
            df["referenced_datafield_id"],
            df["referenced_datafield_name"])
    .distinct()
)
vtv_count = var_to_var.count()
print("\n" + "=" * 65)
print(f"[CHECK 2] Variables referencing another VARIABLE (var→var): {vtv_count:,}",
      "✓" if vtv_count == 0 else "✗ ACTION NEEDED")
if vtv_count > 0:
    display(var_to_var.limit(20))

# ── 5. Variables with no dict linkage ────────────────────────────────────────
null_ref = df.filter(
    (F.col("entry_type") == "VARIABLE") &
    F.col("referenced_datafield_id").isNull()
).count()
null_pct = 100 * null_ref / var_rows if var_rows > 0 else 0
print("\n" + "=" * 65)
print(f"[INFO] VARIABLE rows with NULL referenced_datafield_id: {null_ref:,} ({null_pct:.1f}%)")
print(f"       These variables have no traceable dict linkage (formula-only or unresolved).")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("SUMMARY")
print(f"  True duplicates (Doc+type+entry_id+dict_item) : {true_duplicates:,}  {'✓ PASS' if true_duplicates == 0 else '✗ FAIL'}")
print(f"  Var→var unresolved chains                     : {vtv_count:,}  {'✓ PASS' if vtv_count == 0 else '✗ FAIL'}")
print(f"  Variables with multiple distinct dict items   : {var_rows - distinct_var_entries:,}  (expected — not a defect)")
print(f"  Variables with no dict linkage                : {null_ref:,} ({null_pct:.1f}%)")
